# Лабораторная работа №4. Построение ETL-конвейеров с использованием Dask

**Вариант:** 18

**Цель работы:** получить практические навыки работы с библиотекой Dask для построения базовых ETL-конвейеров (Extract, Transform, Load) при обработке больших массивов данных, не помещающихся в оперативную память. Изучить принципы «ленивых вычислений» (lazy evaluation), управление памятью и визуализацию ориентированных ациклических графов (DAG).

---

## Настройка среды и инициализация Dask

In [ ]:
# Установка Dask с полным набором зависимостей и графов
!pip install "dask[complete]" graphviz altair --quiet

import dask.dataframe as dd
from dask.distributed import Client
from dask.diagnostics import ProgressBar
import dask.delayed as delayed
import os
import pandas as pd
import altair as alt
from datetime import datetime
import re

# Инициализация клиента Dask (Оптимизированные настройки)
client = Client(n_workers=2, threads_per_worker=2, processes=True)
client

## Задание 4.1. Построение ETL-пайплайна средствами Dask

### Шаг 1. Extract (Извлечение данных)
Монтирование Google Drive и «ленивая» загрузка датасета `Austin, TX House Listings.zip`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Путь к файлу на Google Drive (укажите ваш путь)
file_path = '/content/drive/MyDrive/austin_house_listings.zip' 

# Чтение CSV файла из архива
try:
    df = dd.read_csv(file_path, compression='zip', blocksize=None)
    print("Датасет успешно загружен (лениво).")
except Exception as e:
    print(f"Ошибка при загрузке: {e}")

# Просмотр структуры и метаданных датасета
df

### Шаг 2. Transform (Трансформация и очистка данных)
Профилирование качества данных, визуализация с помощью Altair и удаление разреженных столбцов.

In [ ]:
# Подсчет пропущенных значений (построение графа)
missing_values = df.isnull().sum()

# Вычисление процента пропусков
mysize = df.index.size
missing_count = ((missing_values / mysize) * 100)

# Запуск реальных вычислений с использованием ProgressBar
with ProgressBar():
    missing_count_percent = missing_count.compute()

print("\nПроцент пропущенных значений по столбцам:")
print(missing_count_percent)

# Визуализация профиля пропусков с помощью Altair
missing_df = missing_count_percent.reset_index()
missing_df.columns = ['Column', 'Percentage']

chart = alt.Chart(missing_df).mark_bar().encode(
    x=alt.X('Column', sort='-y', title='Название столбца'),
    y=alt.Y('Percentage', title='Процент пропусков (%)'),
    color=alt.condition(
        alt.datum.Percentage > 60,
        alt.value('red'),      
        alt.value('steelblue') 
    ),
    tooltip=['Column', 'Percentage']
).properties(
    title='Профиль качества данных (пропущенные значения)',
    width=600,
    height=300
)
chart.display()

# Формирование списка столбцов для удаления (> 60% пропусков)
columns_to_drop = list(missing_count_percent[missing_count_percent > 60].index)
print("\nУдаляемые столбцы:", columns_to_drop)

# Ленивое удаление столбцов
df_dropped = df.drop(columns=columns_to_drop)

# Проверка результата
df_dropped.head()

### Шаг 3. Load (Загрузка / Сохранение результатов)
Сохранение в формате Parquet.

In [ ]:
output_path = 'cleaned_austin_listings.parquet'
df_dropped.to_parquet(output_path, engine='pyarrow')

print(f"Данные успешно сохранены в {output_path}")

## Задание 4.2. Визуализация направленных ациклических графов (DAG)

In [ ]:
def increment(i):
    return i + 1

def add(x, y):
    return x + y

x = delayed(increment)(10)
y = delayed(increment)(20)
z = delayed(add)(x, y)

print("Граф вычислений для простой цепочки:")
z.visualize()

In [ ]:
print("Результат вычисления DAG:", z.compute())

### 4.2.2. Визуализация многоуровневого (сложного) DAG

In [ ]:
data = [10, 20, 30, 40, 50]
layer1 = [delayed(increment)(i) for i in data]

def square(x):
    return x ** 2

layer2 = [delayed(square)(j) for j in layer1]
total = delayed(sum)(layer2)

print("Граф вычислений для многоуровневой структуры:")
total.visualize()

In [ ]:
print("Итоговый результат сложного DAG:", total.compute())

## #5 Аналитика (Интерактивные графики Altair)
В этой части мы проведем углубленный анализ очищенных данных.

In [ ]:
import pandas as pd
import altair as alt
from datetime import datetime
import re

# ===================================
# 1. Загрузка и предобработка данных (только первые 10000 строк для аналитики)
# ===================================
df_sample = dd.read_parquet('cleaned_austin_listings.parquet').head(10000)

# Очистка дат (если есть колонка с датой продажи, например 'latest_saledate')
# Если такой колонки нет, можно использовать другие признаки для аналитики
date_column = 'latest_saledate'
if date_column in df_sample.columns:
    df_sample[date_column] = pd.to_datetime(df_sample[date_column], errors='coerce')
    df_sample = df_sample.dropna(subset=[date_column])

    # Добавляем вспомогательные колонки
    df_sample['Issue Year'] = df_sample[date_column].dt.year
    df_sample['Issue Month'] = df_sample[date_column].dt.month
    df_sample['Issue DayOfWeek'] = df_sample[date_column].dt.day_name()
    
    print("Колонки дат успешно добавлены.")

# ===================================
# 2. Визуализация: Распределение цен по годам постройи
# ===================================
if 'latestPrice' in df_sample.columns and 'yearBuilt' in df_sample.columns:
    chart1 = alt.Chart(df_sample).mark_circle(size=60).encode(
        x=alt.X('yearBuilt:Q', title='Год постройки'),
        y=alt.Y('latestPrice:Q', title='Цена ($)'),
        color='city:N',
        tooltip=['city', 'latestPrice', 'yearBuilt']
    ).properties(
        title='Зависимость цены от года постройки и города',
        width=700,
        height=400
    ).interactive()
    chart1.display()

# ===================================
# 3. Визуализация: Количество объявлений по месяцам
# ===================================
if 'Issue Month' in df_sample.columns:
    month_counts = df_sample['Issue Month'].value_counts().reset_index()
    month_counts.columns = ['Month', 'Count']
    
    chart2 = alt.Chart(month_counts).mark_bar().encode(
        x=alt.X('Month:O', title='Месяц продажи'),
        y=alt.Y('Count:Q', title='Количество продаж'),
        color=alt.value('orange')
    ).properties(
        title='Динамика продаж по месяцам',
        width=600,
        height=300
)
    chart2.display()